In [1]:
# Ячейка 1 - Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("Библиотеки загружены!")

Библиотеки загружены!


In [2]:
# Ячейка 2 - Загрузка и подготовка данных
df = pd.read_csv('../../data/aruba/aruba.csv', 
                  header=None,
                  sep=',',
                  names=['date', 'time', 'sensor', 'value'])

# Парсим дату
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='mixed', errors='coerce')
df = df.dropna(subset=['datetime'])

# Оставляем только ON события
df = df[df['value'] == 'ON'].copy()

# Создаём признаки
df['hour'] = df['datetime'].dt.hour
df['minute'] = df['datetime'].dt.minute
df['weekday'] = df['datetime'].dt.weekday
df['is_weekend'] = (df['weekday'] >= 5).astype(int)

# Кодируем сенсоры в числа
le = LabelEncoder()
df['sensor_id'] = le.fit_transform(df['sensor'])

print(f"Размер после фильтрации: {df.shape}")
print(f"\nКлассы сенсоров:")
for i, name in enumerate(le.classes_):
    print(f"  {i}: {name}")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/aruba/aruba.csv'

In [4]:
df = pd.read_csv('../data/aruba/aruba.csv', 
                  header=None,
                  sep=',',
                  names=['date', 'time', 'sensor', 'value'])

In [5]:
print(f"Размер: {df.shape}")
print(f"\nКлассы сенсоров:")
for i, name in enumerate(le.classes_):
    print(f"  {i}: {name}")

Размер: (1602821, 4)

Классы сенсоров:


NameError: name 'le' is not defined

In [6]:
# Создаём признаки и кодируем сенсоры
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='mixed', errors='coerce')
df = df.dropna(subset=['datetime'])
df = df[df['value'] == 'ON'].copy()

df['hour'] = df['datetime'].dt.hour
df['minute'] = df['datetime'].dt.minute
df['weekday'] = df['datetime'].dt.weekday
df['is_weekend'] = (df['weekday'] >= 5).astype(int)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['sensor_id'] = le.fit_transform(df['sensor'])

print(f"Размер после фильтрации: {df.shape}")
print(f"\nКлассы сенсоров:")
for i, name in enumerate(le.classes_):
    print(f"  {i}: {name}")

Размер после фильтрации: (797981, 10)

Классы сенсоров:
  0: Bathroom
  1: Bedroom
  2: DiningRoom
  3: GuestRoom
  4: Kitchen
  5: LivingRoom
  6: LoungeChair
  7: OtherRoom
  8: OutsideDoor
  9: WorkArea


In [7]:
# Создаём последовательности для LSTM
SEQUENCE_LENGTH = 10  # используем последние 10 событий для предсказания

features = ['sensor_id', 'hour', 'minute', 'weekday', 'is_weekend']
data = df[features].values

X, y = [], []
for i in range(len(data) - SEQUENCE_LENGTH):
    X.append(data[i:i+SEQUENCE_LENGTH])
    y.append(data[i+SEQUENCE_LENGTH][0])  # предсказываем следующий сенсор

X = np.array(X)
y = np.array(y)

print(f"Форма X: {X.shape}")
print(f"Форма y: {y.shape}")
print(f"\nПример последовательности X[0]:")
print(X[0])
print(f"\nЦелевой сенсор y[0]: {y[0]} = {le.classes_[y[0]]}")

Форма X: (797971, 10, 5)
Форма y: (797971,)

Пример последовательности X[0]:
[[ 1  0  3  3  0]
 [ 1  2 32  3  0]
 [ 1  3 42  3  0]
 [ 1  3 49  3  0]
 [ 1  4 14  3  0]
 [ 1  4 14  3  0]
 [ 1  4 34  3  0]
 [ 1  4 34  3  0]
 [ 1  5 40  3  0]
 [ 1  5 40  3  0]]

Целевой сенсор y[0]: 1 = Bedroom


In [8]:
# Разделяем на train и test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

# Сохраняем данные для обучения модели
np.save('../data/X_train.npy', X_train)
np.save('../data/X_test.npy', X_test)
np.save('../data/y_train.npy', y_train)
np.save('../data/y_test.npy', y_test)

# Сохраняем классы сенсоров
import pickle
with open('../data/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("\nДанные сохранены!")
print(f"Train размер: {len(X_train)}")
print(f"Test размер: {len(X_test)}")

Train: (638376, 10, 5)
Test: (159595, 10, 5)

Данные сохранены!
Train размер: 638376
Test размер: 159595
